In [0]:
%sh wget -c https://current.geneontology.org/ontology/go.obo -P ./programs/

In [0]:
%pip install goatools #https://github.com/tanghaibao/goatools

In [0]:
import subprocess
import os
import pandas as pd
import pickle
import shutil
import sys
from goatools.obo_parser import GODag
from goatools.base import download_go_basic_obo
import io
import glob
import numpy as np

wkdir = os.path.dirname(os.getcwd())

def get_go_descendants(parent_GO_id:str, obo_file = 'programs/go-basic.obo', direct=False, verbose=True):

    #import obo file
    try:
        go_dag = GODag(obo_file)
        if verbose:
            print(f"   Successfully parsed {len(go_dag)} GO terms.")
    except Exception as e:
        print(f"\nERROR: Could not parse GO OBO file. \nDetails: {e}")
        sys.exit(1)
    
    #check if parent GO term exists
    if verbose:
        print(f"\nQuerying descendants for the target term: {parent_GO_id}")
    
    if parent_GO_id not in go_dag:
        print(f"Error: Target GO ID '{parent_GO_id}' not found in the Gene Ontology.")
        return
    
    target_term = go_dag[parent_GO_id]
    if verbose:
        print(f"   Target Term Name: {target_term.name} ({target_term.namespace})")
    
    #get descendants. If direct is True only return direct children
    if direct == True:
        descendant_ids = target_term.children
    else:
        descendant_ids = target_term.get_all_children()
    
    if verbose:
        print(f"   Direct Children (non-recursive): {len(target_term.children)} terms.")
        print(f"   Total Descendants (recursive): {len(target_term.get_all_children())} terms.\n")
    
    #put this in a dict
    descendants = {parent_GO_id:{"name":target_term.name, "level":target_term.level, "depth":target_term.depth}}
    #descendants = {}
    for go_id in sorted(list(descendant_ids)):
        term_obj = go_dag[go_id]
        descendants[go_id] = {"name":term_obj.name, "level":term_obj.level, "depth":term_obj.depth}

    return descendants

def parse_gff_to_dataframe(file_path) -> pd.DataFrame:
    """Parses a GFF file specified by its file path into a pandas DataFrame."""
    data = []
    fixed_columns = ['seqid', 'source', 'type', 'start', 'end', 'score', 'strand', 'phase', 'attributes_raw']

    try:
        with open(file_path, 'r', encoding='utf-8') as content_stream:
            for line in content_stream:
                if line.startswith('#'):
                    continue
                parts = line.strip().split('\t')
                #check valid gff feature
                if len(parts) != 9:
                    continue
                #Extract fixed columns
                row = dict(zip(fixed_columns[:8], parts[:8]))

                #convert start/end to integers
                try:
                    row['start'] = int(row['start'])
                    row['end'] = int(row['end'])
                except ValueError:
                    pass

                #store the original attributes
                attribute_string = parts[8]
                row['attributes_raw'] = attribute_string
                
                #parse 9th column
                attributes = {}
                for part in attribute_string.split(';'):
                    part = part.strip()
                    if not part:
                        continue

                    if '=' in part:
                        key, value = part.split('=', 1)
                        value = value.strip().replace('%20', ' ').replace('%3B', ';').replace('%3D', '=')
                        attributes[key] = value.strip('"')

                #merge fixed columns and parsed attributes
                row.update(attributes)
                data.append(row)
    except FileNotFoundError:
        print(f"Error: The GFF file was not found at path '{file_path}'.")
        return pd.DataFrame()
    except Exception as e:
        print(f"An error occurred during file parsing: {e}")
        return pd.DataFrame()

    df = pd.DataFrame(data)

    #reorder columns
    if not df.empty:
        preferred_cols = fixed_columns[:8] + [
            'ID', 'Name', 'Parent', 'Dbxref', 'gbkey', 'locus_tag', 'product'
        ]
        
        #discard non-existent columns
        existing_cols = [col for col in preferred_cols if col in df.columns]
        remaining_cols = sorted([col for col in df.columns if col not in existing_cols and col not in fixed_columns])
        final_cols = existing_cols + remaining_cols + ['attributes_raw']
        df = df[final_cols]
        
    return df

def is_taxa_downloaded(taxon):
    if os.path.isdir(f"{wkdir}/Functions/refs/{taxon}/ncbi_dataset/data"):
        if os.path.isfile(f"{wkdir}/Functions/refs/{taxon}.zip") == False:
            return True
        else:
            print(f"Partially downloaded {taxon} will start again")
            os.remove(f"{wkdir}/Functions/refs/{taxon}.zip")
            if os.path.isdir(f"{wkdir}/Functions/refs/{taxon}/ncbi_dataset/data"):
                shutil.rmtree(f"{wkdir}/Functions/refs/{taxon}")
    else:
        return False
    
def swap_col_names(df, n_terms):
    cols = df.columns.tolist()
    non_go_cols = ['genus', 'num_genomes', "reference taxa level", "taxon label used"]
    new_names = {}
    for col in cols:
        if col not in non_go_cols:
            new_names[col] = f"{n_terms[col]['name']}({col})"
    print()
    print(new_names)
    return df.rename(columns=new_names)
    
class Genus:
    """Data class for storing downloaded info for genus references."""
    def __init__(self, genus_name):
        self.g = genus_name
        self.taxon = genus_name
        if "Candidatus" in genus_name:
            self.taxon = genus_name.replace("Candidatus ", "")
        self.success = False
        self.level = False
        self.g_error = False
        self.f_error = False
    
    def assign_taxon(self, df):
        self.k = results.loc[results['Genus'] == self.g, 'Kingdom'].item()
        self.p = results.loc[results['Genus'] == self.g, 'Phylum'].item()
        self.c = results.loc[results['Genus'] == self.g, 'Class'].item()
        self.o = results.loc[results['Genus'] == self.g, 'Order'].item()
        self.f = results.loc[results['Genus'] == self.g, 'Family'].item()

def download_references(genus:Genus):
    """Downloads references for a given genus. Assumes you will only look at genus or family level for now."""
    #if the reference for this genus has already been downloaded skip
    if genus.success == True or is_taxa_downloaded(genus.taxon) == True:
        print(f"    ✅ Already downloaded {genus.g}.")
        genus.success = True
        return genus
    
    global manual
    if genus.g in manual.keys() or genus.level == "manual":
        genus.level = "manual"
        genus.taxon = manual[genus.g]
        #checks to see if manual level has already been downloaded by another genus
        if is_taxa_downloaded(genus.taxon) == True:
            genus.success = True
            return genus

    #Set the level to genus if False or Family if it was genus
    elif genus.level == "genus":
        genus.level ="family"
        genus.taxon = genus.f
        #checks to see if family level has already been downloaded by another genus
        if is_taxa_downloaded(genus.taxon) == True:
            genus.success = True
            return genus
    elif genus.level == False:
        genus.level = "genus"
    elif genus.level == "family":
        return genus
    else:
        raise ValueError(f"{genus.level} is an invalid level")

    #Download the genus references for whatever level is set above
    bash_command = f"./programs/datasets download genome taxon '{genus.taxon}' --annotated --reference --assembly-level complete --include gff3 --filename ./refs/{genus.taxon}.zip && unzip ./refs/{genus.taxon}.zip -d ./refs/{genus.taxon} && rm ./refs/{genus.taxon}.zip"
    print(f"    Downloading references for genus {genus.g} at {genus.level} level.")
    try:
        result = subprocess.run(bash_command, shell=True, check=True, text=True, capture_output=True)
        print(f"    ✅ Successfully processed")
        genus.success = True

    except subprocess.CalledProcessError as e:
        print(f"    ❌ Command failed for {genus.taxon}.")
        if genus.level == "genus":
            genus.g_error = e.stderr
        elif genus.level == "family":
            genus.f_error = e.stderr
        else:
            genus.f_error = f"manual error: {e.stderr}"
    
    return genus


In [0]:
#Gets a list of all taxa used
results = pd.read_csv(f"{wkdir}/Output/results_summary_XGB.csv")
genera_names = results['Genus'].tolist()

os.makedirs(f"{wkdir}/Functions/refs", exist_ok=True)

genera = {}

for genus_name in genera_names:
    genus = Genus(genus_name)
    genus.assign_taxon(results)
    genera[genus_name] = genus

with open(f'{wkdir}/Functions/refs/genera.pkl', 'wb') as handle:
    pickle.dump(genera, handle, protocol=-1)

In [0]:
manual = { # = no complete reference genomes availble for that genus and family
    'ADurb.Bin063-1':'1852926', #Verrucomicrobia bacterium ADurb.Bin063 (species, taxid: 1852926, verrucomicrobia)
    'ADurb.Bin118':'1852928', #Verrucomicrobia bacterium ADurb.Bin118 (species, taxid: 1852928, verrucomicrobia)
    'AR15':'1579373', #archaeon GW2011_AR15 (species, taxid: 1579373, archaea)
    'Acidicapsa':'Acidobacteriaceae', #3 reference genomes aren't complete, using 'Acidobacteriaceae' as family level
    'Aliterella':'', #No complete reference genomes at genus or family level (NCBI lists family as Aliterellaceae)
    'Bauldia':'119042', #manually added taxonomy id for Hyphomicrobiales incertae sedis
    'Blvii28 wastewater-sludge group':'Rikenellaceae', #midas field guide lists this aka 'Acetobacteroides' (no references) and family aka 'Rikenellaceae'
    'Candidatus Nitrosoarchaeum':'Nitrosopumilaceae', #no complete references at genus level and 'Nitrosopumilaceae' listed as family in NCBI
    'Candidatus Omnitrophus':'67812', #no complete references at genus level and 'Candidatus Omnitrophota' listed as family in NCBI
    'Candidatus Xiphinematobacter':'', #no reference genomes for genera and no family listed for NCBI
    'Chroococcidiopsis PCC-6712':'Chroococcidiopsis',
    'Elstera':'Rhodospirillaceae', #no genus level references but there are references for old family name 'Rhodospirillaceae'
    'FukuN18 freshwater group':'', #no reference genome for NCBI name 'uncultured bacterium FukuN18' and no family in NCBI
    'Fusibacter':'', #no reference genome for genus level or NCBI family 'Eubacteriales Family XII. Incertae Sedis'
    'GW2011_GWC1_47_15':'', #no referenc genome for any 'Candidatus Amesbacteria bacterium' species and no fmaily level in NCBI
    'Hassallia':'', #no reference for genus level or NCBI family 'Tolypothrichaceae'
    'Inquilinus':'Rhodospirillaceae', #no genus level references but there are references for old family name 'Rhodospirillaceae'
    'JGI 0001001-H03':'', #species under 'Acidobacteria bacterium JGI 0001001-H03' has no reference genome and NCBI lists no family
    'Labrys':'204476', #Labrys (genus, taxid: 204476, a-proteobacteria) and has 1 complete genome
    'Marine Benthic Group D and DHVEG-1':'', #no genus or family labels in NCBI and order 'Candidatus Thermoprofundales' has no complete genomes
    'Methylobacter':'Methylococcaceae', #no genus level references but there are references for old fmaily name 'Methylococcaceae'
    'Methyloglobulus':'Methylococcaceae', #no genus level references but there are references for old fmaily name 'Methylococcaceae'
    'OLB13':'', #NCBI labels for genus ('Flexicrinis') and family ('Flexifilaceae') have no complete reference genomes
    'Ohtaekwangia':'Fulvivirgaceae', #no genus label references but NCBI name for family 'Fulvivirgaceae' does
    'Phormidesmis ANT.LACV5.1':'Leptolyngbyaceae', #no genus label references for Phormidesmis found but NCBI family 'Leptolyngbyaceae' does
    'Planctomicrobium':'Planctomycetaceae', #no genus label references for Phormidesmis found but NCBI family 'Planctomycetaceae' does
    'Pleurocapsa PCC-7319':'', #no genus level or family reference level for SILVA or NCBI labels for that genus
    'Rivicola':'', #genus name doesnt match NCBI name but no references anyway
    'WY65':'Acidobacterium', #Acidobacterium sp. WY65 has 'Acidobacterium' listed as genus in NCBI and has one complete reference
    'pLW-20':'' #LPSN synonym for 'Methylococcaceae' which is an NCBI family level taxonomy
}

with open(f'{wkdir}/Functions/refs/genera.pkl', 'rb') as file:
    genera = pickle.load(file)

for genus_name in genera:
    genera[genus_name] = download_references(genera[genus_name])

for genus_name in genera:
    genera[genus_name] = download_references(genera[genus_name])

for genus_name in genera:
    genera[genus_name] = download_references(genera[genus_name])

with open(f'{wkdir}/Functions/refs/genera.pkl', 'wb') as handle:
    pickle.dump(genera, handle, protocol=-1)

In [0]:
with open(f'{wkdir}/Functions/refs/genera.pkl', 'rb') as file:
    genera = pickle.load(file)

n_terms = get_go_descendants('GO:0071941', obo_file = 'programs/go.obo')

genera_all_n = pd.DataFrame()

for genus_name in genera.keys():
    genus = genera[genus_name]
    if genus.success == False:
        genera_all_n = pd.concat([genera_all_n, pd.DataFrame({"genus":[genus.g],"num_genomes":[0], "reference taxa level":["fail"], "taxon label used":[genus.taxon]})])
        continue
    genus_all = pd.DataFrame()
    paths = glob.glob(f"{wkdir}/Functions/refs/{genus.taxon}/ncbi_dataset/data/GC*")
    for p in paths:
        ref = os.path.basename(p)
        df = parse_gff_to_dataframe(f"{wkdir}/Functions/refs/{genus.taxon}/ncbi_dataset/data/{ref}/genomic.gff")
        proteins = df[df['type'] == "CDS"]
        proteins = proteins.dropna(axis=1, how='all')
        try:
            proteins['Ontology_list'] = proteins['Ontology_term'].str.split(',')
        except KeyError as e:
            print(f"KeyError for '{genus.taxon}-{ref}': no {e} column found.")
            continue
        df_exploded = proteins.explode('Ontology_list')
        df_dummies = pd.get_dummies(df_exploded, columns=['Ontology_list'], prefix='', prefix_sep='')
        df_encoded = df_dummies.groupby('seqid').sum(numeric_only=True)
        reference_all = df_encoded.max(axis=0)
        ref_df = reference_all.to_frame().T.drop(columns=['start','end'])
        ref_df['ref'] = ref
        cols = ref_df.columns.tolist()
        cols.remove('ref')
        ref_df = ref_df[['ref']+cols]
        genus_all = pd.concat([genus_all, ref_df])
    numeric_cols = genus_all.select_dtypes(include=np.number).columns
    genus_all[numeric_cols] = np.where(genus_all[numeric_cols] <= 1, 0, 1)
    num_genomes = len(genus_all)
    genus_all_n = pd.DataFrame({"genus":[genus.g],"num_genomes":[num_genomes], "reference taxa level":[genus.level], "taxon label used":[genus.taxon]})
    for go in n_terms.keys():
        try:
            genus_all_n[go] = (genus_all[go].sum(numeric_only=True)/num_genomes)*100
        except KeyError:
            genus_all_n[go] = 0
    genus.n_go = genus_all_n.copy()
    genera_all_n = pd.concat([genera_all_n, genus_all_n])


with open(f'{wkdir}/Functions/refs/genera.pkl', 'wb') as handle:
    pickle.dump(genera, handle, protocol=-1)

In [0]:
with open(f'{wkdir}/Functions/refs/genera.pkl', 'rb') as file:
    genera = pickle.load(file)

n_terms = get_go_descendants('GO:0071941', obo_file = 'programs/go.obo')
genera_all_n = pd.DataFrame()

for genus_name in genera.keys():
    genus = genera[genus_name]
    try:
        genera_all_n = pd.concat([genera_all_n, genus.n_go])
    except AttributeError as e:
        continue

genera_all_n.fillna(0, inplace=True)

for n_term in list(n_terms.keys()):
    if n_term == 'GO:0071941':
        continue
    descendants = get_go_descendants(n_term, obo_file = 'programs/go.obo', verbose=False)
    if len(descendants) > 1:
        print(f"Adding descendants of {n_term} to parent term.")
        for descendant in list(descendants.keys()):
            if descendant != n_term:
                try:
                    genera_all_n[n_term] = genera_all_n[n_term] + genera_all_n[descendant]
                    genera_all_n.drop(descendant, axis=1, inplace=True)
                    print(f"    Adding {descendant}.")
                except KeyError:
                    print(f"    {descendant} not found.")


n_cols = list(n_terms.keys())
n_cols = [item for item in n_cols if item in genera_all_n.columns.tolist()]
genera_all_n[n_cols] = np.where(genera_all_n[n_cols] > 0, 1, genera_all_n[n_cols])

genera_all_n = swap_col_names(genera_all_n, n_terms)

display(genera_all_n)

genera_all_n.to_csv('n_go_pa.csv', index=False)